# COMSOL向けEEDFテーブル生成 (広域E/Nスイープ + Ar*混入対応)

純Ar + 準安定Ar*(比率調整可)の電子EEDFを広いE/N範囲(0.5〜50000 Td)で計算し、
COMSOL PlasmaインターフェースのInterpolation関数(2引数, Spreadsheet形式)として
エクスポートするノートブック。

各E/Nに対してエネルギーグリッド上限 `eps_max` を**自動調整**する
(EEPF末端値がピークの1e-6以下になるまで拡大、過剰なら縮小)。
このため `--eps-max-evを増やしてください` エラーは発生しない。

**注意事項**

- 同梱のAr断面積データは**1000 eVまで**。それ以上は最終値で一定外挿されるため、
  おおむね 2000 Td 超(裾が1 keVを大きく超える領域)の結果は定量的には参考値。
  高E/Nを本気で使う場合は1 keV以上をカバーするLXCatデータを`bp.parse_lxcat`で与えること。
- 低E/N側(< 5 Td)は緩和が遅く、1点あたり1〜2分かかる。
- 50000 Tdでは平均エネルギーが約1.7 keVに達する(相対論効果は未考慮)。

In [ ]:
import math
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import boltzpmp as bp

warnings.simplefilter("ignore")  # 裾の警告はソルバ側で自動処理するため抑制

In [ ]:
# ============ パラメータ ============
METASTABLE_FRACTION = 1e-5     # Ar*(4s)の比率 (0で純Ar)
FIELDS_TD = np.geomspace(0.5, 50000.0, 11)  # 計算するE/N [Td]
PRESSURE_PA = 133.0
GAS_TEMPERATURE_K = 273.0

N_CELLS = 300                  # エネルギーセル数 (d_eps = eps_max / N_CELLS)
N_THETA = 48
TOL = 1e-5
MAX_STEPS = 2_000_000          # 低E/Nは百万ステップ超が必要

OUTPUT = Path("comsol_eedf_argon_sweep.txt")
MOBILITY_OUTPUT = Path("comsol_mobility_argon_sweep.txt")
N_EXPORT_POINTS = 400          # エクスポート時の共通エネルギーグリッド点数

print("fields [Td]:", np.round(FIELDS_TD, 2))

In [ ]:
# ============ eps_max自動調整ソルバ ============
TAIL_MAX = 1e-6    # EEPF(eps_max)/peak がこれを超えたらeps_max不足 -> 拡大
TAIL_MIN = 1e-12   # これを下回ったらeps_max過剰 -> 縮小して分解能を確保


def eps_max_guess(EN_Td):
    """純Ar実測に基づく必要eps_maxの初期推定 (安全率1.3)。"""
    x = math.log10(EN_Td)
    return max(20.0, 1.3 * 10 ** (0.1805 * x * x - 0.0346 * x + 0.9967))


def solve_adaptive(mixture, EN_Td, *, n_cells=N_CELLS, n_theta=N_THETA,
                   tol=TOL, max_steps=MAX_STEPS, max_retries=8):
    """tail条件を満たすまでeps_maxを自動調整して解く。"""
    em = eps_max_guess(EN_Td)
    shrunk = False
    for _ in range(max_retries):
        solver = bp.PMSolver(mixture, eps_max_eV=em, d_eps_eV=em / n_cells,
                             n_theta=n_theta)
        result = solver.solve_dc(EN_Td=EN_Td, tol=tol, max_steps=max_steps,
                                 check_every=200)
        if not result.converged:
            raise RuntimeError(f"{EN_Td:g} Td: {result.n_steps}ステップで未収束")
        tail = float(result.extra["eepf_tail_ratio"])
        if tail > TAIL_MAX:
            em *= 1.6           # 裾が収まっていない -> グリッド拡大
            continue
        if tail < TAIL_MIN and not shrunk:
            # グリッドが過剰に広い -> EEPFがピークの1e-8に落ちる位置まで縮小
            eepf = np.asarray(result.eepf, float)
            ratio = eepf / eepf.max()
            idx = np.nonzero(ratio > 1e-8)[0][-1]
            em_new = 1.3 * float(np.asarray(result.energy_grid, float)[idx])
            if em_new < 0.7 * em:
                em = max(em_new, 5.0)
                shrunk = True
                continue
        return result, em
    raise RuntimeError(f"{EN_Td:g} Td: eps_max自動調整が{max_retries}回で収束せず")

In [ ]:
# ============ スイープ実行 ============
mixture = bp.load_argon(
    metastable_fraction=METASTABLE_FRACTION,
    p_Pa=PRESSURE_PA,
    T_K=GAS_TEMPERATURE_K,
)

results = []
for EN in sorted(FIELDS_TD):
    t0 = time.time()
    result, em = solve_adaptive(mixture, EN)
    results.append(result)
    print(
        f"E/N={EN:10.2f} Td  eps_max={em:9.0f} eV  "
        f"<e>={result.mean_energy:10.3f} eV  "
        f"vd={result.drift_velocity:9.3e} m/s  "
        f"steps={result.n_steps:>8}  ({time.time() - t0:.1f}s)"
    )

In [ ]:
# ============ 可視化 ============
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5))

for r in results:
    ax1.plot(r.energy_grid, r.eepf, label=f"{r.extra['EN_Td']:g} Td")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_ylim(1e-12, None)
ax1.set_xlabel("electron energy [eV]")
ax1.set_ylabel("EEPF [eV$^{-3/2}$]")
ax1.legend(fontsize=8)
ax1.set_title(f"EEPF (Ar* fraction = {METASTABLE_FRACTION:g})")

fields = [r.extra["EN_Td"] for r in results]
means = [r.mean_energy for r in results]
ax2.loglog(fields, means, "o-")
ax2.set_xlabel("E/N [Td]")
ax2.set_ylabel("mean energy [eV]")
ax2.set_title("mean electron energy")
ax2.grid(True, which="both", alpha=0.3)

mobilities = [r.drift_velocity / (r.extra["EN_Td"] * 1e-21) for r in results]
ax3.loglog(means, mobilities, "s-", color="tab:red")
ax3.set_xlabel("mean energy [eV]")
ax3.set_ylabel("reduced mobility $\\mu N$ [1/(V m s)]")
ax3.set_title("reduced electron mobility")
ax3.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============ COMSOL Spreadsheet形式でエクスポート ============
def trapezoid(values, coordinates):
    widths = np.diff(coordinates)
    return float(np.sum(0.5 * (values[:-1] + values[1:]) * widths))


def export_comsol(results, output, n_points=N_EXPORT_POINTS):
    """全E/Nの結果を共通エネルギーグリッドへ再標本化して3列で書き出す。

    列: 電子エネルギー[eV] / 平均エネルギー[eV] / EEPF[eV^-3/2]
    グリッドを共通化することでCOMSOLの2引数補間が確実に読める
    矩形データになる。
    """
    em_global = max(float(r.mesh.eps_max_eV) for r in results)
    grid = np.concatenate(([0.0], np.geomspace(0.05, em_global, n_points - 1)))
    rows, summaries = [], []
    for r in sorted(results, key=lambda r: r.mean_energy):
        e = np.asarray(r.energy_grid, float)
        f = np.maximum(np.asarray(r.eepf, float), 0.0)
        logf = np.log(np.maximum(f, 1e-300))
        fi = np.exp(np.interp(grid, e, logf, left=logf[0], right=-700.0))
        fi[fi < 1e-290] = 0.0
        fi /= trapezoid(np.sqrt(grid) * fi, grid)   # ∫√eps f deps = 1
        mean_energy = trapezoid(grid ** 1.5 * fi, grid)
        rows.append(np.column_stack((grid, np.full_like(grid, mean_energy), fi)))
        summaries.append((float(r.extra["EN_Td"]), mean_energy))
    header = (
        "COMSOL Spreadsheet data for a two-argument EEDF interpolation function\n"
        "Argument units in COMSOL: V, V; function unit: V^(-3/2)\n"
        "electron_energy_eV\tmean_electron_energy_eV\teedf_eV^-3/2"
    )
    np.savetxt(output, np.vstack(rows), fmt="%.17e", delimiter="\t",
               header=header, comments="% ", encoding="utf-8")
    return summaries


summaries = export_comsol(results, OUTPUT)
print(f"wrote {OUTPUT.resolve()}")
for EN, me in summaries:
    print(f"  E/N={EN:10.2f} Td  exported mean energy={me:10.4f} eV")

In [ ]:
# ============ 換算電子移動度のエクスポート ============
def export_mobility(results, output):
    """平均エネルギーの関数として換算移動度 muN [1/(V m s)] を書き出す。

    muN = v_drift / (E/N)。COMSOLでは1引数のInterpolation関数として読み込み、
    引数単位 V、関数単位 1/(V*m*s) を設定する。
    """
    rows = sorted(
        (
            (float(r.mean_energy),
             float(r.drift_velocity) / (float(r.extra["EN_Td"]) * 1e-21))
            for r in results
        ),
        key=lambda row: row[0],
    )
    header = (
        "COMSOL Spreadsheet data for reduced electron mobility muN\n"
        "Argument unit in COMSOL: V; function unit: 1/(V*m*s)\n"
        "mean_electron_energy_eV\treduced_mobility_1_per_Vms"
    )
    np.savetxt(output, np.asarray(rows), fmt="%.17e", delimiter="\t",
               header=header, comments="% ", encoding="utf-8")
    return rows


mobility_rows = export_mobility(results, MOBILITY_OUTPUT)
print(f"wrote {MOBILITY_OUTPUT.resolve()}")
for me, muN in mobility_rows:
    print(f"  <e>={me:10.4f} eV  muN={muN:.5e} 1/(V m s)")

## COMSOLへの読み込み方

1. **Global Definitions → Interpolation** を追加し、Data source: *File*、
   Data format: *Spreadsheet*、Number of arguments: **2** に設定して
   上で書き出したファイルを読み込む。
2. 引数の単位を **V, V**、関数の単位を **V^(-3/2)** に設定する。
3. Plasmaインターフェースの電子エネルギー分布関数で、この補間関数を指定する。

第1引数が電子エネルギー、第2引数が平均電子エネルギー。COMSOLは局所平均
エネルギーに応じて2つのEEDF曲線間を補間する。スイープのE/N点数を増やすほど
補間精度が上がる(`FIELDS_TD`を調整)。

### 換算電子移動度

1. もう1つ **Interpolation** を追加し、Number of arguments: **1** で
   移動度ファイルを読み込む。
2. 引数の単位を **V**、関数の単位を **1/(V\*m\*s)** に設定する。
3. Plasmaインターフェースの *Electron mobility* で
   **Reduced electron mobility** を選び、この補間関数を平均エネルギーの
   関数として指定する(電子拡散係数・エネルギー移動度・エネルギー拡散係数は
   アインシュタインの関係から自動計算させるのが標準)。
